# ZINDI Arabic Multi-Task Classification

Predict six targets from Arabic news text with a multi-head transformer ensemble.

| Target | Type | Classes |
|---|---|---|
| Qatar Related | binary | 0 / 1 |
| Geography | multiclass | G-1..G-12, NO_GEOGRAPHY |
| Politics & Conflict | multiclass | PC-1, PC-2, NO_POLITICS |
| Health & Wellbeing | multiclass | H-1, H-2, NO_HEALTH |
| Science | multiclass | SC-1..SC-3, NO_SCIENCE |
| Sports | multiclass | SP-1..SP-4, NO_SPORTS |

**Metric:** mean of per-column weighted F1 across all 6 targets.

**Leakage policy (compliant).** Train and Test contain some identical article texts. These are
reported as a *diagnostic only*; training labels are **never transferred** to test rows. All
predictions come from models trained on `Train.csv`.

**Throughput:** variable-length sequences with dynamic per-batch padding, length-bucketed training
batches, larger batches with LR scaling, and tuned DataLoader workers.

**Pipeline**
1. Arabic preprocessing + train de-duplication (majority vote) for honest CV.
2. TF-IDF + Logistic Regression baseline (5-fold OOF).
3. Multi-head transformers (AraBERTv02, MARBERTv2, CAMeLBERT-Mix, AraBERT-large), 5-fold OOF, cached per model.
4. Per-head ensemble weight search + per-head prior/temperature calibration on OOF.
5. Structural domain constraint (every article has >=1 non-`NO_` domain).
6. Optional CV-validated pseudo-labeling (off by default).
7. Compliance note + submission.


In [ ]:
%pip install -q "transformers>=4.40,<5" 

In [ ]:
import copy
import hashlib
import math
import os
import pickle
import random
import re
import time
import warnings
from collections import Counter
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42


def seed_everything(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


seed_everything()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device:", DEVICE, "| torch", torch.__version__)

CANDIDATE_DIRS = [
    Path("/content"),
    Path.cwd(),
    Path("/Users/ahadhussain/Desktop/Scratch projects/googleaihackathon"),
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "Train.csv").exists()), CANDIDATE_DIRS[0])
OUTPUT_DIR = (DATA_DIR / "submissions") if str(DATA_DIR).startswith("/content") else Path("/tmp/submissions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

ID_COLUMN = "ID"
TEXT_COLUMN = "text"
QATAR_COLUMN = "Qatar Related"
REMAINING_TARGETS = ["Geography", "Politics & Conflict", "Health & Wellbeing", "Science", "Sports"]
TARGET_COLUMNS = [QATAR_COLUMN] + REMAINING_TARGETS
DOMAIN_TARGETS = ["Politics & Conflict", "Health & Wellbeing", "Science", "Sports"]

# ----------------------------- encoders -----------------------------
MODEL_NAMES = [
    "aubmindlab/bert-base-arabertv02",
    "UBC-NLP/MARBERTv2",
    "CAMeL-Lab/bert-base-arabic-camelbert-mix",
    "aubmindlab/bert-large-arabertv02",
]
MODEL_REVISIONS = {}

# ----------------------------- throughput config -----------------------------
DEFAULT_BATCH = 32
MODEL_BATCH_SIZE = {"aubmindlab/bert-large-arabertv02": 16}
BASE_BATCH = 16
BASE_LR = 2e-5
LR_SCALING = "sqrt"          # "sqrt" | "linear" | "none"
USE_BUCKET_SAMPLER = True
NUM_WORKERS = 4 if DEVICE.type == "cuda" else 0
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
ATTN_IMPL = "sdpa"           # set None to disable

# ----------------------------- training config -----------------------------
MAX_LEN = 512
N_FOLDS = 5
EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
USE_CLASS_WEIGHTS = False
LABEL_SMOOTHING = 0.0
MAX_CLASS_WEIGHT = 5.0
HEAD_LOSS_WEIGHTS = {"Geography": 2.0}   # per-task loss weights (Geography is the hardest head)
SEEDS = [42, 1337]                       # multi-seed averaging per encoder

# ----------------------------- experiment switches -----------------------------
RUN_TIMING_PROBE = True
RUN_AB_TESTS = True
RUN_PSEUDO = False
PSEUDO_THRESH = 0.90
PRIOR_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0]


def lr_for_batch(batch):
    if LR_SCALING == "sqrt":
        return BASE_LR * (batch / BASE_BATCH) ** 0.5
    if LR_SCALING == "linear":
        return BASE_LR * (batch / BASE_BATCH)
    return BASE_LR


CONFIG_TAG = hashlib.md5(
    f"{MAX_LEN}|{DEFAULT_BATCH}|{sorted(MODEL_BATCH_SIZE.items())}|{LR_SCALING}|{EPOCHS}|"
    f"{ATTN_IMPL}|{USE_CLASS_WEIGHTS}|{LABEL_SMOOTHING}|{sorted(HEAD_LOSS_WEIGHTS.items())}|"
    f"{SEEDS}|{USE_BUCKET_SAMPLER}".encode()
).hexdigest()[:8]
print("CONFIG_TAG:", CONFIG_TAG)
print("batch default", DEFAULT_BATCH, "| large", MODEL_BATCH_SIZE, "| lr_for_batch base =", round(lr_for_batch(DEFAULT_BATCH), 6))
print("workers", NUM_WORKERS, "| bucket", USE_BUCKET_SAMPLER, "| attn", ATTN_IMPL)
print("seeds", SEEDS, "| head loss weights", HEAD_LOSS_WEIGHTS)
if DEVICE.type == "cuda":
    print("cuda:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0),
          f"| VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("cuda: False -> running on", DEVICE.type, "(GPU stays idle during TF-IDF; it engages in the transformer cells)")

In [ ]:
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061a\u064b-\u0652]")
NON_TEXT = re.compile(r"[^\w\s\u0600-\u06ff]")
WHITESPACE = re.compile(r"\s+")


def preprocess_arabic(text):
    text = str(text)
    text = ARABIC_DIACRITICS.sub("", text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي").replace("ة", "ه")
    return WHITESPACE.sub(" ", NON_TEXT.sub(" ", text)).strip()


def load_data():
    train = pd.read_csv(DATA_DIR / "Train.csv")
    test = pd.read_csv(DATA_DIR / "Test.csv")
    for c in TARGET_COLUMNS:
        train[c] = train[c].astype(str)
    return train, test


def score_predictions(actual, predicted, columns=TARGET_COLUMNS):
    scores = {c: f1_score(actual[c], predicted[c], average="weighted") for c in columns}
    scores["mean"] = float(np.mean(list(scores.values())))
    return scores


def make_submission(test_frame, predictions, output_path):
    sub = test_frame[[ID_COLUMN]].copy()
    sub[TARGET_COLUMNS] = predictions[TARGET_COLUMNS].to_numpy()
    sub.to_csv(output_path, index=False)
    return sub


train_df, test_df = load_data()
train_df["processed_text"] = train_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)
test_df["processed_text"] = test_df[TEXT_COLUMN].fillna("").map(preprocess_arabic)
print("train:", train_df.shape, "| test:", test_df.shape)

In [ ]:
# ---------- DIAGNOSTIC ONLY: exact text overlap between train and test ----------
# Competition rule: identical records must NOT be used to transfer training labels to
# test predictions. We only report the overlap for transparency; no labels are stored or copied.
train_texts = set(train_df["processed_text"])
n_overlap = int(test_df["processed_text"].isin(train_texts).sum())
print(f"[diagnostic only] exact train/test text overlap: {n_overlap}/{len(test_df)} "
      f"({n_overlap / len(test_df):.1%}) -- labels NOT transferred")

# ---------- de-duplicate train (majority vote) for honest CV ----------
agg = {c: (c, lambda s: s.value_counts().idxmax()) for c in TARGET_COLUMNS}
agg["n_dupes"] = (ID_COLUMN, "size")
dedup = train_df.groupby("processed_text").agg(**agg).reset_index()
conflicts = sum(
    1 for _, g in train_df.groupby("processed_text")
    if len(g) > 1 and any(g[c].nunique() > 1 for c in TARGET_COLUMNS)
)
print(f"Train deduped: {len(train_df)} -> {len(dedup)} unique texts")
print("duplicate groups:", int((dedup["n_dupes"] > 1).sum()),
      "| groups with conflicting labels:", conflicts)

In [ ]:
label_encoders = {c: LabelEncoder().fit(train_df[c]) for c in TARGET_COLUMNS}
NUM_CLASSES = {c: len(label_encoders[c].classes_) for c in TARGET_COLUMNS}
for c in TARGET_COLUMNS:
    print(f"{c:20s} {NUM_CLASSES[c]:2d}  {list(label_encoders[c].classes_)}")

y_all = {c: label_encoders[c].transform(dedup[c].values) for c in TARGET_COLUMNS}
PRIORS = {}
for c in TARGET_COLUMNS:
    p = np.bincount(y_all[c], minlength=NUM_CLASSES[c]).astype(np.float64) + 1e-6
    PRIORS[c] = p / p.sum()


def domain_key(row):
    for c in DOMAIN_TARGETS:
        if not str(row[c]).startswith("NO_"):
            return row[c]
    return "NONE"


strat = (dedup[QATAR_COLUMN].values + "_" + dedup.apply(domain_key, axis=1).values)
counts = Counter(strat)
rare = {k for k, v in counts.items() if v < N_FOLDS}
strat = np.array([f"q{k.split('_')[0]}" if k in rare else k for k in strat])
print("\nstrat classes:", len(set(strat)), "| min count:", min(Counter(strat).values()))

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(dedup, strat))
for i, (tr, va) in enumerate(folds):
    print(f"fold {i}: train {len(tr)} val {len(va)}")

## 1. TF-IDF + Logistic Regression baseline (5-fold OOF)

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
tfidf_oof = {c: np.zeros((len(dedup), NUM_CLASSES[c]), dtype=np.float32) for c in TARGET_COLUMNS}

for tr_idx, va_idx in folds:
    Xtr = tfidf.fit_transform(dedup.iloc[tr_idx]["processed_text"])
    Xva = tfidf.transform(dedup.iloc[va_idx]["processed_text"])
    for c in TARGET_COLUMNS:
        clf = LogisticRegression(max_iter=2000, C=4.0)
        clf.fit(Xtr, y_all[c][tr_idx])
        proba = np.zeros((len(va_idx), NUM_CLASSES[c]), dtype=np.float32)
        proba[:, clf.classes_] = clf.predict_proba(Xva)
        tfidf_oof[c][va_idx] = proba

tfidf_preds = {c: label_encoders[c].inverse_transform(tfidf_oof[c].argmax(1)) for c in TARGET_COLUMNS}
tfidf_scores = score_predictions(dedup[TARGET_COLUMNS], pd.DataFrame(tfidf_preds))
print("TF-IDF + LogReg OOF")
for c in TARGET_COLUMNS:
    print(f"  {c:20s} {tfidf_scores[c]:.4f}")
print(f"  {'MEAN':20s} {tfidf_scores['mean']:.4f}")

Xtr_full = tfidf.fit_transform(dedup["processed_text"])
Xte = tfidf.transform(test_df["processed_text"])
tfidf_test_probs = {}
for c in TARGET_COLUMNS:
    clf = LogisticRegression(max_iter=2000, C=4.0)
    clf.fit(Xtr_full, y_all[c])
    proba = np.zeros((len(test_df), NUM_CLASSES[c]), dtype=np.float32)
    proba[:, clf.classes_] = clf.predict_proba(Xte)
    tfidf_test_probs[c] = proba

## 2. Multi-head transformer

One shared encoder + six linear heads. Head+tail truncation keeps the start of the article (topic)
and the end (source/author line). Sequences are variable length; padding happens per batch.
Encoders are cached per model and per config, so a new encoder or config only trains what is missing.


In [ ]:
def encode_sequences(tokenizer, texts, max_len=MAX_LEN):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        enc = tokenizer(list(texts), add_special_tokens=False)["input_ids"]
    cls_id, sep_id = tokenizer.cls_token_id, tokenizer.sep_token_id
    body = max_len - 2
    seqs = []
    for ids in enc:
        ids = list(ids)
        if len(ids) > body:
            head = body * 3 // 4
            ids = ids[:head] + ids[-(body - head):]
        seqs.append(np.array([cls_id] + ids + [sep_id], dtype=np.int64))
    return seqs


class NewsDataset(Dataset):
    def __init__(self, seqs, labels=None):
        self.seqs = seqs
        self.labels = labels

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        item = {"input_ids": torch.tensor(self.seqs[i], dtype=torch.long)}
        if self.labels is not None:
            item["labels"] = torch.tensor([self.labels[c][i] for c in TARGET_COLUMNS], dtype=torch.long)
        return item


def collate_batch(features, pad_id):
    maxlen = max(len(f["input_ids"]) for f in features)
    ids = torch.full((len(features), maxlen), pad_id, dtype=torch.long)
    attn = torch.zeros((len(features), maxlen), dtype=torch.long)
    for i, f in enumerate(features):
        L = len(f["input_ids"])
        ids[i, :L] = f["input_ids"]
        attn[i, :L] = 1
    batch = {"input_ids": ids, "attention_mask": attn}
    if "labels" in features[0]:
        batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch


def make_collate(pad_id):
    return lambda feats: collate_batch(feats, pad_id)


def dl_kwargs():
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        kw["persistent_workers"] = PERSISTENT_WORKERS
        kw["prefetch_factor"] = PREFETCH_FACTOR
    return kw


class BucketBatchSampler:
    def __init__(self, lengths, batch_size, shuffle=True, seed=SEED, pool_factor=50):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.pool = max(batch_size * pool_factor, batch_size)
        self.epoch = 0

    def set_epoch(self, e):
        self.epoch = e

    def __iter__(self):
        order = np.argsort(self.lengths, kind="stable")
        batches = []
        for i in range(0, len(order), self.pool):
            pool = order[i:i + self.pool].copy()
            if self.shuffle:
                np.random.default_rng(self.seed + self.epoch * 1000 + i).shuffle(pool)
            for j in range(0, len(pool), self.batch_size):
                batches.append(pool[j:j + self.batch_size].tolist())
        if self.shuffle:
            np.random.default_rng(self.seed + self.epoch).shuffle(batches)
        for b in batches:
            yield b

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)


class MultiTaskModel(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super().__init__()
        kwargs = {"revision": MODEL_REVISIONS[model_name]} if model_name in MODEL_REVISIONS else {}
        if ATTN_IMPL:
            try:
                self.encoder = AutoModel.from_pretrained(model_name, attn_implementation=ATTN_IMPL, **kwargs)
            except Exception as e:
                print(f"  attn_implementation={ATTN_IMPL} unavailable ({type(e).__name__}); using default")
                self.encoder = AutoModel.from_pretrained(model_name, **kwargs)
        else:
            self.encoder = AutoModel.from_pretrained(model_name, **kwargs)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({c: nn.Linear(h, num_classes[c]) for c in TARGET_COLUMNS})

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        out = self.dropout(out)
        return {c: self.heads[c](out) for c in TARGET_COLUMNS}


@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    out = {c: [] for c in TARGET_COLUMNS}
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        logits = model(ids, mask)
        for c in TARGET_COLUMNS:
            out[c].append(torch.softmax(logits[c].float(), 1).cpu().numpy())
    return {c: np.concatenate(v, 0) for c, v in out.items()}


def compute_class_weights(y_dict, tr_idx):
    weights = {}
    for c in TARGET_COLUMNS:
        cnt = np.bincount(y_dict[c][tr_idx], minlength=NUM_CLASSES[c]).astype(np.float32)
        cnt[cnt == 0] = 1
        w = len(tr_idx) / (NUM_CLASSES[c] * cnt)
        weights[c] = torch.tensor(np.clip(w, 0, MAX_CLASS_WEIGHT), dtype=torch.float32)
    return weights

In [ ]:
def train_fold_model(model_name, train_seqs, tr_idx, va_idx, y_dict,
                     epochs=EPOCHS, use_class_weights=USE_CLASS_WEIGHTS,
                     label_smoothing=LABEL_SMOOTHING, batch_size=None, fold_idx=None,
                     pad_id=0, seed=SEED, head_loss_weights=None):
    seed_everything(seed + int(tr_idx[0]) % 1000)
    hw = head_loss_weights if head_loss_weights is not None else HEAD_LOSS_WEIGHTS
    head_w = {c: float(hw.get(c, 1.0)) for c in TARGET_COLUMNS}
    head_w_sum = sum(head_w.values())
    bs = batch_size or MODEL_BATCH_SIZE.get(model_name, DEFAULT_BATCH)
    ds_tr = NewsDataset([train_seqs[i] for i in tr_idx], {c: y_dict[c][tr_idx] for c in TARGET_COLUMNS})
    ds_va = NewsDataset([train_seqs[i] for i in va_idx], {c: y_dict[c][va_idx] for c in TARGET_COLUMNS})
    collate = make_collate(pad_id)
    kw = dl_kwargs()
    sampler = None
    if USE_BUCKET_SAMPLER:
        sampler = BucketBatchSampler([len(s) for s in ds_tr.seqs], bs, shuffle=True, seed=seed)
        dl_tr = DataLoader(ds_tr, batch_sampler=sampler, collate_fn=collate, **kw)
    else:
        dl_tr = DataLoader(ds_tr, batch_size=bs, shuffle=True, collate_fn=collate, **kw)
    dl_va = DataLoader(ds_va, batch_size=bs * 2, shuffle=False, collate_fn=collate, **kw)

    model = MultiTaskModel(model_name, NUM_CLASSES).to(DEVICE)
    cw = {c: w.to(DEVICE) for c, w in compute_class_weights(y_dict, tr_idx).items()} if use_class_weights else None
    lr = lr_for_batch(bs)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total = len(dl_tr) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best = (-1.0, None, None)
    for ep in range(epochs):
        if sampler is not None:
            sampler.set_epoch(ep)
        model.train()
        running = 0.0
        for batch in dl_tr:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(ids, mask)
                loss = sum(
                    head_w[c] * F.cross_entropy(logits[c], labels[:, i], weight=(cw[c] if cw else None),
                                                label_smoothing=label_smoothing)
                    for i, c in enumerate(TARGET_COLUMNS)
                ) / head_w_sum
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()

        va_p = predict_probs(model, dl_va)
        va_pred = {c: label_encoders[c].inverse_transform(va_p[c].argmax(1)) for c in TARGET_COLUMNS}
        sc = score_predictions(dedup.iloc[va_idx].reset_index(drop=True), pd.DataFrame(va_pred))
        tag = f"fold {fold_idx} " if fold_idx is not None else ""
        print(f"  [{model_name.split('/')[-1]}] {tag}ep {ep + 1}: "
              f"loss={running / len(dl_tr):.4f} meanWF1={sc['mean']:.4f}")
        if sc["mean"] > best[0]:
            best = (sc["mean"], copy.deepcopy(model.state_dict()), va_p)
    model.load_state_dict(best[1])
    return model, best[1], best[2]


def train_model(model_name, folds, test_texts, epochs=EPOCHS, max_len=MAX_LEN,
                use_class_weights=USE_CLASS_WEIGHTS, label_smoothing=LABEL_SMOOTHING,
                seed=SEED):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, revision=MODEL_REVISIONS.get(model_name))
    print(f"[{model_name}] seed {seed} | encoding {len(dedup)} train / {len(test_texts)} test texts ...")
    train_seqs = encode_sequences(tokenizer, dedup["processed_text"].tolist(), max_len)
    test_seqs = encode_sequences(tokenizer, test_texts, max_len)
    pad_id = tokenizer.pad_token_id
    bs = MODEL_BATCH_SIZE.get(model_name, DEFAULT_BATCH)
    dl_te = DataLoader(NewsDataset(test_seqs), batch_size=bs * 2, shuffle=False,
                       collate_fn=make_collate(pad_id), **dl_kwargs())

    oof = {c: np.zeros((len(dedup), NUM_CLASSES[c]), dtype=np.float32) for c in TARGET_COLUMNS}
    test_fold_probs = {c: [] for c in TARGET_COLUMNS}
    for fi, (tr_idx, va_idx) in enumerate(folds):
        model, state, va_p = train_fold_model(model_name, train_seqs, tr_idx, va_idx, y_all,
                                              epochs, use_class_weights, label_smoothing,
                                              fold_idx=fi, pad_id=pad_id, seed=seed)
        for c in TARGET_COLUMNS:
            oof[c][va_idx] = va_p[c]
        te_p = predict_probs(model, dl_te)
        for c in TARGET_COLUMNS:
            test_fold_probs[c].append(te_p[c])
        print(f"  [{model_name.split('/')[-1]}] fold {fi} done")
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    test_probs = {c: np.mean(test_fold_probs[c], axis=0) for c in TARGET_COLUMNS}
    return oof, test_probs


def train_full(model_name, texts, y_dict, epochs=EPOCHS, max_len=MAX_LEN, seed=SEED):
    seed_everything(seed)
    head_w = {c: float(HEAD_LOSS_WEIGHTS.get(c, 1.0)) for c in TARGET_COLUMNS}
    head_w_sum = sum(head_w.values())
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, revision=MODEL_REVISIONS.get(model_name))
    seqs = encode_sequences(tokenizer, texts, max_len)
    pad_id = tokenizer.pad_token_id
    bs = MODEL_BATCH_SIZE.get(model_name, DEFAULT_BATCH)
    kw = dl_kwargs()
    if USE_BUCKET_SAMPLER:
        sampler = BucketBatchSampler([len(s) for s in seqs], bs, shuffle=True, seed=SEED)
        dl = DataLoader(NewsDataset(seqs, y_dict), batch_sampler=sampler,
                        collate_fn=make_collate(pad_id), **kw)
    else:
        dl = DataLoader(NewsDataset(seqs, y_dict), batch_size=bs, shuffle=True,
                        collate_fn=make_collate(pad_id), **kw)
    model = MultiTaskModel(model_name, NUM_CLASSES).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr_for_batch(bs), weight_decay=WEIGHT_DECAY)
    total = len(dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(total * WARMUP_RATIO), total)
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    for ep in range(epochs):
        model.train()
        for batch in dl:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(ids, mask)
                loss = sum(head_w[c] * F.cross_entropy(logits[c], labels[:, i])
                           for i, c in enumerate(TARGET_COLUMNS)) / head_w_sum
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
    return model

In [ ]:
def safe_name(model_name):
    return re.sub(r"[^A-Za-z0-9]+", "_", model_name)


def cache_path(model_name, seed):
    return OUTPUT_DIR / f"probs_cache_{safe_name(model_name)}_s{seed}_{CONFIG_TAG}.pkl"


model_oof, model_test_probs, models_trained = {}, {}, []
for m in MODEL_NAMES:
    oof_seeds, test_seeds = [], []
    for sd in SEEDS:
        cp = cache_path(m, sd)
        if cp.exists():
            with open(cp, "rb") as f:
                d = pickle.load(f)
            oof_seeds.append(d["oof"])
            test_seeds.append(d["test_probs"])
            print(f"[cache] loaded {m} seed {sd}")
            continue
        try:
            oof, tp = train_model(m, folds, test_df["processed_text"].tolist(), seed=sd)
            with open(cp, "wb") as f:
                pickle.dump({"oof": oof, "test_probs": tp}, f)
            oof_seeds.append(oof)
            test_seeds.append(tp)
        except Exception as e:
            print(f"FAILED {m} seed {sd}: {type(e).__name__}: {e}")
    if oof_seeds:
        model_oof[m] = {c: np.mean([o[c] for o in oof_seeds], 0) for c in TARGET_COLUMNS}
        model_test_probs[m] = {c: np.mean([t[c] for t in test_seeds], 0) for c in TARGET_COLUMNS}
        models_trained.append(m)
        preds = {c: label_encoders[c].inverse_transform(model_oof[m][c].argmax(1)) for c in TARGET_COLUMNS}
        sc = score_predictions(dedup[TARGET_COLUMNS], pd.DataFrame(preds))
        print(f"=> {m} OOF meanWF1 = {sc['mean']:.4f} ({len(oof_seeds)} seed(s), geo={sc['Geography']:.4f})")
print("\nmodels trained:", models_trained)

## 3. Timing probe

Runs one epoch on fold 0 of the primary encoder and reports wall-clock, tokens/sec and peak GPU
memory, so throughput changes can be measured instead of guessed.

In [ ]:
if RUN_TIMING_PROBE:
    PRIMARY = MODEL_NAMES[0]
    tok = AutoTokenizer.from_pretrained(PRIMARY, revision=MODEL_REVISIONS.get(PRIMARY))
    seqs_p = encode_sequences(tok, dedup["processed_text"].tolist())
    tr_idx, va_idx = folds[0]
    bs = MODEL_BATCH_SIZE.get(PRIMARY, DEFAULT_BATCH)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    mdl, _, _ = train_fold_model(PRIMARY, seqs_p, tr_idx, va_idx, y_all, epochs=1,
                                 fold_idx=0, pad_id=tok.pad_token_id)
    dt = time.time() - t0
    n_tok = int(sum(len(seqs_p[i]) for i in tr_idx))
    mem = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    print(f"probe: batch={bs} lr={lr_for_batch(bs):.2e} bucket={USE_BUCKET_SAMPLER} attn={ATTN_IMPL} ")
    print(f"probe: 1 epoch {dt:.1f}s | {n_tok / dt:,.0f} tokens/s | peak GPU {mem:.2f} GB")
    del mdl
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

## 4. Fold-0 A/B tests (reporting only)

Compares class weights, label smoothing, `MAX_LEN` and epochs on fold 0 of the primary encoder.
Results are informational — change the config cell if a variant clearly wins.

In [ ]:
if RUN_AB_TESTS:
    PRIMARY = MODEL_NAMES[0]
    tokenizer = AutoTokenizer.from_pretrained(PRIMARY, revision=MODEL_REVISIONS.get(PRIMARY))
    tr_idx, va_idx = folds[0]
    va_true = dedup.iloc[va_idx].reset_index(drop=True)

    def ab_run(name, max_len=MAX_LEN, epochs=EPOCHS, use_class_weights=USE_CLASS_WEIGHTS,
               label_smoothing=LABEL_SMOOTHING, head_loss_weights=None):
        seqs = encode_sequences(tokenizer, dedup["processed_text"].tolist(), max_len)
        mdl, _, va_p = train_fold_model(PRIMARY, seqs, tr_idx, va_idx, y_all,
                                        epochs, use_class_weights, label_smoothing,
                                        pad_id=tokenizer.pad_token_id,
                                        head_loss_weights=head_loss_weights)
        preds = {c: label_encoders[c].inverse_transform(va_p[c].argmax(1)) for c in TARGET_COLUMNS}
        sc = score_predictions(va_true, pd.DataFrame(preds))
        print(f"AB {name:24s} meanWF1={sc['mean']:.4f}  (geo={sc['Geography']:.4f})")
        del mdl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return sc

    ab_results = {}
    ab_results["baseline"] = ab_run("baseline")
    ab_results["geo_weight_off"] = ab_run("geo_weight_off", head_loss_weights={})
    ab_results["class_weights"] = ab_run("class_weights", use_class_weights=True)
    ab_results["label_smoothing_0.05"] = ab_run("label_smoothing_0.05", label_smoothing=0.05)
    ab_results["max_len_384"] = ab_run("max_len_384", max_len=384)
    ab_results["epochs_4"] = ab_run("epochs_4", epochs=4)
    print("\nbest fold-0 variant:", max(ab_results, key=lambda k: ab_results[k]["mean"]))

## 5. Ensemble + per-head tuning

For each target we greedily choose which components to average (they differ per target), then tune a
per-head prior/temperature correction — the main lever for the imbalanced Geography head. Both are
fitted on OOF predictions only.

In [ ]:
def blend(prob_dicts):
    return {c: np.mean([d[c] for d in prob_dicts], axis=0) for c in TARGET_COLUMNS}


def decode(probs):
    return pd.DataFrame({c: label_encoders[c].inverse_transform(probs[c].argmax(1)) for c in TARGET_COLUMNS})


def head_f1(probs_c, c):
    return f1_score(dedup[c].values, label_encoders[c].inverse_transform(probs_c.argmax(1)), average="weighted")


def score_probs(probs):
    return score_predictions(dedup[TARGET_COLUMNS], decode(probs))


def greedy_head(comps, c, eps=1e-5):
    names = list(comps)
    start = max(names, key=lambda n: head_f1(comps[n][c], c))
    chosen = [start]
    cur = comps[start][c].astype(np.float64).copy()
    cur_f = head_f1(cur, c)
    while True:
        best_name, best_f, best_cur = None, cur_f, None
        for n in names:
            if n in chosen:
                continue
            cand = (cur * len(chosen) + comps[n][c]) / (len(chosen) + 1)
            f = head_f1(cand, c)
            if f > best_f + eps:
                best_name, best_f, best_cur = n, f, cand
        if best_name is None:
            break
        chosen.append(best_name)
        cur, cur_f = best_cur, best_f
    return chosen


oof_components = {"tfidf": tfidf_oof}
test_components = {"tfidf": tfidf_test_probs}
for m in models_trained:
    oof_components[m] = model_oof[m]
    test_components[m] = model_test_probs[m]

oof_variants = {k: v for k, v in oof_components.items()}
test_variants = {k: v for k, v in test_components.items()}
if len(models_trained) > 1:
    oof_variants["transformer_blend"] = blend([model_oof[m] for m in models_trained])
    test_variants["transformer_blend"] = blend([model_test_probs[m] for m in models_trained])
oof_variants["all_blend"] = blend(list(oof_components.values()))
test_variants["all_blend"] = blend(list(test_components.values()))

head_choice = {c: greedy_head(oof_components, c) for c in TARGET_COLUMNS}
oof_variants["per_head"] = {c: np.mean([oof_components[n][c] for n in head_choice[c]], 0) for c in TARGET_COLUMNS}
test_variants["per_head"] = {c: np.mean([test_components[n][c] for n in head_choice[c]], 0) for c in TARGET_COLUMNS}


def build_variant_test(components, name):
    if name == "per_head":
        return {c: np.mean([components[n][c] for n in head_choice[c]], 0) for c in TARGET_COLUMNS}
    if name == "all_blend":
        return blend(list(components.values()))
    if name == "transformer_blend":
        return blend([components[m] for m in models_trained])
    return components[name]


for c in TARGET_COLUMNS:
    print(f"{c:20s} uses {[n.split('/')[-1] if '/' in n else n for n in head_choice[c]]}")

variant_scores = {k: score_probs(v) for k, v in oof_variants.items()}
for k, sc in sorted(variant_scores.items(), key=lambda x: -x[1]["mean"]):
    print(f"{k:28s} meanWF1={sc['mean']:.4f}")

CHOSEN = max(variant_scores, key=lambda k: variant_scores[k]["mean"])
print(f"\nChosen ensemble: {CHOSEN} (OOF {variant_scores[CHOSEN]['mean']:.4f})")
oof_probs = oof_variants[CHOSEN]
test_probs = test_variants[CHOSEN]

best_alpha = {}
for c in TARGET_COLUMNS:
    base_f = head_f1(oof_probs[c], c)
    a_best, f_best = 0.0, base_f
    for a in PRIOR_GRID:
        adj = oof_probs[c].astype(np.float64) / np.power(PRIORS[c], a)
        f = head_f1(adj, c)
        if f > f_best + 1e-6:
            a_best, f_best = a, f
    best_alpha[c] = a_best
    if a_best != 0.0:
        oof_probs[c] = oof_probs[c].astype(np.float64) / np.power(PRIORS[c], a_best)
        test_probs[c] = test_probs[c].astype(np.float64) / np.power(PRIORS[c], a_best)
print("\nprior alphas:", {c: best_alpha[c] for c in TARGET_COLUMNS})
final_oof_score = score_probs(oof_probs)
print(f"after prior calibration OOF meanWF1 = {final_oof_score['mean']:.4f}")

## 6. Structural domain constraint

Every article has at least one non-`NO_` target among the 4 domain heads. If a row is predicted all-`NO_`,
promote the domain head/class with the highest non-`NO_` probability. Applied only if it improves OOF.

In [ ]:
NO_INDEX = {}
for c in DOMAIN_TARGETS:
    classes = [str(x) for x in label_encoders[c].classes_]
    NO_INDEX[c] = next(i for i, x in enumerate(classes) if x.startswith("NO_"))


def constrained_indices(probs):
    preds = {c: probs[c].argmax(1).astype(np.int64).copy() for c in TARGET_COLUMNS}
    n = len(preds[TARGET_COLUMNS[0]])
    for i in range(n):
        if all(preds[c][i] == NO_INDEX[c] for c in DOMAIN_TARGETS):
            best_c, best_j, best_p = None, None, -np.inf
            for c in DOMAIN_TARGETS:
                p = probs[c][i].astype(np.float64).copy()
                p[NO_INDEX[c]] = -np.inf
                j = int(p.argmax())
                if p[j] > best_p:
                    best_c, best_j, best_p = c, j, p[j]
            preds[best_c][i] = best_j
    return preds


def decode_indices(idxs):
    return pd.DataFrame({c: label_encoders[c].inverse_transform(idxs[c]) for c in TARGET_COLUMNS})


base_sc = score_predictions(dedup[TARGET_COLUMNS], decode(oof_probs))
con_sc = score_predictions(dedup[TARGET_COLUMNS], decode_indices(constrained_indices(oof_probs)))
print(f"base OOF        meanWF1={base_sc['mean']:.4f}")
print(f"constrained OOF meanWF1={con_sc['mean']:.4f}")
USE_CONSTRAINT = con_sc["mean"] > base_sc["mean"]
print("USE_CONSTRAINT =", USE_CONSTRAINT)
final_oof_score = con_sc if USE_CONSTRAINT else base_sc
print(f"final applied OOF meanWF1 = {final_oof_score['mean']:.4f}")

## 7. Geography diagnostics

Pinpoints where the hardest head fails: per-class report, confusion structure, worst pairs and examples.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

GEO = "Geography"
geo_true = dedup[GEO].values
geo_pred = label_encoders[GEO].inverse_transform(oof_probs[GEO].argmax(1))
geo_labels = list(label_encoders[GEO].classes_)

print(classification_report(geo_true, geo_pred, labels=geo_labels, zero_division=0, digits=3))

cm = confusion_matrix(geo_true, geo_pred, labels=geo_labels)
cm_df = pd.DataFrame(cm, index=geo_labels, columns=geo_labels)
cm_norm = cm_df.div(cm_df.sum(1).replace(0, 1), axis=0).round(2)
print("row-normalized confusion (rows = true, cols = predicted):")
display(cm_norm)

conf = []
for i, ti in enumerate(geo_labels):
    for j, tj in enumerate(geo_labels):
        if i != j and cm[i, j] > 0:
            conf.append((int(cm[i, j]), ti, tj))
conf.sort(reverse=True)
print("\ntop confusions (true -> predicted):")
for n, ti, tj in conf[:12]:
    print(f"  {ti:15s} -> {tj:15s} {n}")

wrong = np.where(geo_pred != geo_true)[0]
print(f"\nmisclassified: {len(wrong)}/{len(geo_true)} ; Geography weighted F1 = {head_f1(oof_probs[GEO], GEO):.4f}")
for i in wrong[:6]:
    print(f"  true={geo_true[i]:14s} pred={geo_pred[i]:14s} | {dedup['processed_text'].iloc[i][:110]}")

## 8. Optional: CV-validated pseudo-labeling

Selects confident test rows (all six heads >= `PSEUDO_THRESH`), then checks on CV whether adding them
helps. Set `RUN_PSEUDO = True` in the config cell to enable; it adds a full training pass.

In [ ]:
pseudo_applied = False
if RUN_PSEUDO:
    conf = np.min(np.stack([test_probs[c].max(1) for c in TARGET_COLUMNS], axis=1), axis=1)
    seen = set(dedup["processed_text"])
    keep = (conf >= PSEUDO_THRESH) & (~test_df["processed_text"].isin(seen).values)
    extra_texts = test_df.loc[keep, "processed_text"].tolist()
    extra_labels = {c: label_encoders[c].inverse_transform(test_probs[c][keep].argmax(1))
                    for c in TARGET_COLUMNS}
    print(f"confident pseudo rows: {len(extra_texts)} (threshold {PSEUDO_THRESH})")

    if len(extra_texts) >= 50:
        PRIMARY = models_trained[0] if models_trained else MODEL_NAMES[0]
        tokenizer = AutoTokenizer.from_pretrained(PRIMARY, revision=MODEL_REVISIONS.get(PRIMARY))
        comb_texts = dedup["processed_text"].tolist() + extra_texts
        comb_seqs = encode_sequences(tokenizer, comb_texts)
        pad_id = tokenizer.pad_token_id
        extra_y = {c: label_encoders[c].transform(extra_labels[c]) for c in TARGET_COLUMNS}
        comb_y = {c: np.concatenate([y_all[c], extra_y[c]]) for c in TARGET_COLUMNS}
        extra_idx = np.arange(len(dedup), len(dedup) + len(extra_texts))

        oof_p = {c: np.zeros((len(dedup), NUM_CLASSES[c]), dtype=np.float32) for c in TARGET_COLUMNS}
        for fi, (tr_idx, va_idx) in enumerate(folds):
            mdl, _, va_p = train_fold_model(PRIMARY, comb_seqs, np.concatenate([tr_idx, extra_idx]),
                                            va_idx, comb_y, fold_idx=fi, pad_id=pad_id)
            for c in TARGET_COLUMNS:
                oof_p[c][va_idx] = va_p[c]
            del mdl
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        base_ref = score_probs({c: model_oof[PRIMARY][c] for c in TARGET_COLUMNS})["mean"]
        pseudo_sc = score_probs(oof_p)["mean"]
        print(f"pseudo CV: base={base_ref:.4f} -> pseudo={pseudo_sc:.4f}")

        if pseudo_sc > base_ref:
            model = train_full(PRIMARY, comb_texts, comb_y)
            test_seqs_p = encode_sequences(tokenizer, test_df["processed_text"].tolist())
            bs = MODEL_BATCH_SIZE.get(PRIMARY, DEFAULT_BATCH)
            dl_te_p = DataLoader(NewsDataset(test_seqs_p), batch_size=bs * 2, shuffle=False,
                                 collate_fn=make_collate(pad_id), **dl_kwargs())
            test_components[PRIMARY] = predict_probs(model, dl_te_p)
            test_probs = build_variant_test(test_components, CHOSEN)
            for c in TARGET_COLUMNS:
                if best_alpha.get(c, 0.0) != 0.0:
                    test_probs[c] = test_probs[c].astype(np.float64) / np.power(PRIORS[c], best_alpha[c])
            pseudo_applied = True
            print(f"pseudo-labeling ADOPTED for {PRIMARY}")
        else:
            print("pseudo-labeling rejected (no CV gain)")
    else:
        print("not enough confident pseudo rows; skipping")
print("pseudo_applied =", pseudo_applied)

## 9. Compliance & reproducibility

- **Data:** only the challenge-provided `Train.csv` / `Test.csv`. No external datasets.
- **No leakage:** exact train/test duplicates are never used to transfer labels (the diagnostic above
  is informational only).
- **Open tooling:** open-source packages (torch, transformers, scikit-learn, pandas) and openly
  available pretrained models only; no paid APIs.
- **Reproducibility:** `SEED = 42` and `seed_everything()` are set; the bucketed sampler is seeded per
  epoch, so reruns reproduce. Pin exact model commits via `MODEL_REVISIONS` if required for review.
- Do not share this notebook outside your Zindi team; publish via the challenge platform only.


## 10. Final submission

In [ ]:
if USE_CONSTRAINT:
    test_idx = constrained_indices(test_probs)
else:
    test_idx = {c: test_probs[c].argmax(1).astype(np.int64) for c in TARGET_COLUMNS}
test_pred = decode_indices(test_idx)

FINAL_PATH = OUTPUT_DIR / "submission_multitask.csv"
submission = make_submission(test_df, test_pred, FINAL_PATH)

assert len(submission) == len(test_df)
assert list(submission.columns) == [ID_COLUMN] + TARGET_COLUMNS
assert set(submission[ID_COLUMN]) == set(test_df[ID_COLUMN])
print("Saved:", FINAL_PATH)
display(submission.head())

In [ ]:
print("=== Final OOF (chosen, calibrated) ===")
for c in TARGET_COLUMNS:
    print(f"  {c:20s} {final_oof_score[c]:.4f}")
print(f"  {'MEAN':20s} {final_oof_score['mean']:.4f}")
print(f"Constraint applied: {USE_CONSTRAINT}")
print("Submission:", FINAL_PATH)

In [ ]:
# optional: download from Colab
# from google.colab import files
# files.download(str(FINAL_PATH))